# Optimal Model — HuffPost News Classification

**Strategy:** RoBERTa-base two-phase fine-tuning with:
1. **Label consolidation** 41 → 35 classes (merges near-synonymous categories with F1 < 0.45)
2. **CLS token pooling** (replaces GlobalAveragePooling — CLS is what RoBERTa is trained to use for classification)
3. **Focal loss** (replaces cross-entropy + class weights — focuses training on hard/minority examples)
4. **val_accuracy early stopping** (not val_loss — these diverge on transformer fine-tuning)
5. **Warmup + cosine decay LR schedule** in Phase 2 (replaces flat 2e-5)
6. **Longer Phase 2** (up to 10 epochs with patience=3)

**Baseline to beat:** OPTIMAL v1 DistilBERT — test acc 0.6457 / macro-F1 0.5824

## Background: Milestone 1 — EDA & Problem Framing

### Dataset
The HuffPost News Category dataset contains **200,853 news articles** across **41 categories** with fields: `headline`, `short_description`, `category`, `authors`, `link`, `date`. The team selected this dataset over image classification alternatives due to its real-world NLP applicability.

### Key EDA Findings

**Class imbalance (32× ratio):**
| Category | Count |
|---|---|
| POLITICS (largest) | 32,739 |
| WELLNESS | 17,827 |
| ENTERTAINMENT | 16,058 |
| EDUCATION (smallest) | 1,004 |

**Text length statistics:**
- Headline: mean 9.5 words (std 3.1), range 0–44 words
- Description: mean 19.7 words (std 14.4), range 0–243 words
- Combined (p50 / p95 / p99): 29 / 56 / 68 words → max_length=128 truncates only ~0.1% of samples

**Data quality issues:**
- 6 empty headlines (dropped)
- 19,712 articles (~9.8%) missing `short_description` → used headline-only text
- 1,398 duplicate (headline, category) pairs
- 717 headlines appearing under multiple categories (genuine label ambiguity)

**Category overlap identified (root cause of model confusion):**
```
ARTS & CULTURE  ↔  ARTS  ↔  CULTURE & ARTS
STYLE           ↔  STYLE & BEAUTY
HEALTHY LIVING  ↔  WELLNESS
THE WORLDPOST   ↔  WORLDPOST
PARENTS         ↔  PARENTING
TASTE           ↔  FOOD & DRINK
```

### Preprocessing Pipeline

```python
# 1. Text construction
text = headline.lower() + " [sep] " + short_description.lower()

# 2. Drop near-empty samples
df = df[df['text'].str.len() > 5]

# 3. Stratified 80/10/10 split
train_test_split(..., stratify=labels)

# 4. Class weights (balanced) to address 32× imbalance
compute_class_weight("balanced", classes=np.arange(41), y=train_labels)
# Weight range: 0.150× (POLITICS) – 4.874× (EDUCATION)

# 5. Tokenization
TextVectorization(max_tokens=20000, output_sequence_length=128)
```

### Feature Engineering Decisions

- **Vocabulary cap at 20,000 tokens** — sufficient for news domain; larger vocab didn't improve results
- **Sequence length 128** — preserves 99%+ of samples; no meaningful truncation loss
- **`[sep]` separator token** — concatenates headline and description into single input sequence
- **t-SNE / PCA visualizations** — confirmed no linear separation between categories; overlap validated the need for deep architectures
- **GloVe 100d identified as promising** — EDA showed word frequency alone insufficient to distinguish overlapping categories; pretrained semantic vectors expected to help

### Primary Success Metric: Macro-F1

Accuracy is misleading on a 32× imbalanced 41-class dataset. Macro-F1 treats all classes equally — a model that ignores minority classes (e.g. EDUCATION, GOOD NEWS) is penalized regardless of overall accuracy.


## Background: Milestone 2 — Model Experiments & Key Learnings

### Experiment Summary

Three models were trained on the 41-class dataset. Results in this notebook directly informed the optimal model design.

| Model | Architecture | Test Acc | Macro-F1 | Train Time | Key Decision |
|---|---|---|---|---|---|
| Baseline | Embedding(20k,64) → GAP → Dense(64) → Dense(41) | 0.5832 | — | 4.75 min | Bag-of-words ceiling |
| Custom (BiLSTM) | GloVe(100d, trainable) → BiLSTM(64) → Dense(64) → Dense(41) | 0.6433 | — | 121.7 min | **Pretrained embeddings decisive** |
| DistilBERT | Frozen base → GAP → Dropout(0.3) → Dense(41) (two-phase in TCB) | 0.6457 | 0.5824 | 135.2 min | **Full fine-tuning essential** |

### What Each Experiment Revealed

**Baseline (Embedding + GlobalAveragePooling):**
```python
Embedding(20000, 64) → GlobalAveragePooling1D → Dense(64, relu) → Dense(41, softmax)
# Result: test_acc=0.5832, best_epoch=14, train_time=4.75 min
```
- GAP discards word order — treats each document as a bag of tokens
- Overfitting after epoch 14; early stopping at epoch 19
- Establishes the ceiling for order-insensitive models: ~58% accuracy
- **Learning:** Ordering matters for distinguishing overlapping news categories

**Custom Model (GloVe + BiLSTM):**
```python
Embedding(20000, 100, weights=[glove_matrix], trainable=True)
  → Bidirectional(LSTM(64)) → Dropout(0.3) → Dense(64, relu) → Dropout(0.2) → Dense(41, softmax)
# Result: test_acc=0.6433, best_epoch=4, train_time=121.7 min
```
Three variants were tested before settling on the best:

| Variant | Setup | Test Acc |
|---|---|---|
| V1 | GloVe frozen → GAP → Dense(64) | 0.5289 |
| V2 | GloVe frozen → Dropout(0.4) → GAP → Dense(64) | 0.4550 |
| **V3 (best)** | **GloVe trainable → BiLSTM(64) → Dropout(0.3) → Dense(64) → Dense(41)** | **0.6433** |

- Pretrained GloVe embeddings gave +6.2% over baseline — the single highest-leverage decision at this tier
- **Trainable embeddings essential:** frozen embeddings (V1) underperformed; V2 showed aggressive dropout (0.4) on frozen embeddings actively hurts
- **BiLSTM captures word order** — critical for distinguishing `POLITICS` from `WORLD NEWS`, `WELLNESS` from `HEALTHY LIVING`
- Converged at epoch 4 (fast); val_loss and val_accuracy largely agreed at this model tier

**DistilBERT Fine-tuning:**

The TCB notebook used two-phase fine-tuning (decisive advantage):
```python
# Phase 1: frozen base, train head only
distilbert_base.trainable = False
model.compile(optimizer=Adam(1e-3), ...)   # head: GAP → Dropout(0.3) → Dense(41)
# Phase 2: full fine-tune at controlled LR
distilbert_base.trainable = True
model.compile(optimizer=Adam(2e-5), ...)
# Result: test_acc=0.6457, macro-F1=0.5824, train_time=135.2 min
```

RTF notebook used frozen base only → test_acc=0.6017 (–4.4%). DistilBERT still improving at epoch 10 when stopped — not converged.

**Critical observation — val_loss vs val_accuracy divergence:**
- On the 41-class imbalanced dataset, `val_loss` bottomed out at epoch 2 for the BiLSTM
- `val_accuracy` kept improving until epoch 6
- Monitoring `val_loss` for early stopping prematurely stops training
- **Fix in optimal model: monitor `val_accuracy`, not `val_loss`**

### Per-Class F1 (DistilBERT, 41 classes)

Hardest categories (F1 < 0.40):
```
GOOD NEWS   0.352    IMPACT      0.371    EDUCATION   0.383
```
Easiest categories (F1 > 0.85):
```
STYLE & BEAUTY  0.868    WEDDINGS  0.859    SPORTS  0.844
```

Hardest categories are either minority classes or have ambiguous/overlapping content. This directly motivated the label consolidation (41 → 35) in this notebook.

### Three Improvements Implemented in This Notebook

1. **Label consolidation 41 → 35** — merges the 7 near-synonymous category pairs identified in EDA; removes systematic confusion at the data level
2. **`val_accuracy` early stopping** — replaces `val_loss` monitoring to avoid premature convergence
3. **Warmup + cosine decay LR schedule (Phase 2)** — replaces flat `lr=2e-5`; linear warmup prevents large early gradient updates on the unfrozen base, cosine decay provides smooth annealing


In [ ]:
# ── Package setup ─────────────────────────────────────────────────────────────
# transformers 5.x dropped TF support; 4.47.1 is the last version with TFRobertaModel.
# tf_keras provides the legacy Keras 2 backend required by transformers 4.x.
import subprocess, sys
for pkg in ["tf_keras", "transformers==4.47.1"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)

import os, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tf_keras as keras
from tf_keras import layers

from datasets import load_dataset, DatasetDict, load_from_disk
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report
from transformers import RobertaTokenizerFast, TFRobertaModel

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# ── Hyperparameters ───────────────────────────────────────────────────────────
SEED         = 42
BERT_MAX_LEN = 128
BERT_BATCH   = 32

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 1. Data Preparation

In [ ]:
# ── Load raw dataset ──────────────────────────────────────────────────────────
DISK_PATH = "huffpost_splits"
RAW_PATH  = "huffpost.json"
URL       = "https://huggingface.co/datasets/khalidalt/HuffPost/resolve/main/News_Category_Dataset_v2.json"

if os.path.exists(DISK_PATH):
    huff = load_from_disk(DISK_PATH)
    print("Loaded from disk cache.")
elif os.path.exists(RAW_PATH):
    huff = load_dataset("json", data_files=RAW_PATH, split="train")
    print("Loaded from local JSON.")
else:
    huff = load_dataset("json", data_files=URL, split="train")
    huff.save_to_disk(DISK_PATH)
    print("Downloaded and cached to disk.")

print(f"Raw records: {len(huff):,}")

# ── Label consolidation: 41 → 35 classes ──────────────────────────────────────
# Merges categories that are near-synonymous and caused systematic model confusion.
# Evidence: DistilBERT F1 < 0.45 on all categories within each merge group.
MERGE_MAP = {
    "ARTS & CULTURE": "ARTS",           # F1 overlap: ARTS ↔ ARTS & CULTURE ↔ CULTURE & ARTS
    "CULTURE & ARTS": "ARTS",
    "STYLE":          "STYLE & BEAUTY", # Same category, renamed at some point in dataset
    "HEALTHY LIVING": "WELLNESS",       # Near-identical content and vocabulary
    "THE WORLDPOST":  "WORLDPOST",      # Same publication, different label
    "PARENTS":        "PARENTING",      # Identical audience and content
    "TASTE":          "FOOD & DRINK",   # Taste is the food/drink section of HuffPost
}

def consolidate_label(ex):
    return {"category": MERGE_MAP.get(ex["category"], ex["category"])}

huff2 = huff.map(consolidate_label)

# ── Clean and normalize text ──────────────────────────────────────────────────
def build_text(ex):
    h = (ex.get("headline") or "").strip().lower()
    s = (ex.get("short_description") or "").strip().lower()
    return {"text": (h + " [sep] " + s).strip()}

huff2 = huff2.map(build_text)
before = len(huff2)
huff2  = huff2.filter(lambda ex: len(ex["text"].strip()) > 5)
print(f"Dropped {before - len(huff2)} near-empty samples; {len(huff2):,} remaining.")

# ── Integer-encode labels ─────────────────────────────────────────────────────
huff2       = huff2.class_encode_column("category")
label_names = huff2.features["category"].names
NUM_CLASSES = len(label_names)

print(f"\nClasses after consolidation: {NUM_CLASSES}  (was 41)")
print("Merged categories:")
for src, tgt in MERGE_MAP.items():
    print(f"  {src}  →  {tgt}")

In [ ]:
# ── Stratified 80 / 10 / 10 split ────────────────────────────────────────────
tmp       = huff2.train_test_split(test_size=0.10, seed=SEED, stratify_by_column="category")
train_val = tmp["train"].train_test_split(test_size=1/9, seed=SEED, stratify_by_column="category")
ds = DatasetDict(train=train_val["train"], val=train_val["test"], test=tmp["test"])

print("Split sizes:")
for split, d in ds.items():
    print(f"  {split:6s}: {len(d):>7,} samples")

train_texts  = np.array(ds["train"]["text"])
train_labels = np.array(ds["train"]["category"])
val_texts    = np.array(ds["val"]["text"])
val_labels   = np.array(ds["val"]["category"])
test_texts   = np.array(ds["test"]["text"])
test_labels  = np.array(ds["test"]["category"])

# ── Class weights ─────────────────────────────────────────────────────────────
cw_array     = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_labels)
class_weight = dict(enumerate(cw_array))
print(f"\nClass weight range: {cw_array.min():.3f} – {cw_array.max():.3f}")
print(f"  Highest (minority): {label_names[int(np.argmax(cw_array))]}  ({cw_array.max():.2f}×)")
print(f"  Lowest  (majority): {label_names[int(np.argmin(cw_array))]}  ({cw_array.min():.2f}×)")

## 2. RoBERTa Tokenization

In [ ]:
# ── Tokenize all splits with RoBERTa's BPE tokenizer ────────────────────────
print("Loading tokenizer...")
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

def bert_tokenize(texts, max_len=BERT_MAX_LEN):
    enc = tokenizer(
        list(texts),
        max_length=max_len,
        truncation=True,
        padding="max_length",
        return_tensors="np",
    )
    return enc["input_ids"], enc["attention_mask"]

print("Tokenizing splits (≈10s on GPU machine)...")
t0 = time.time()
tr_ids,  tr_mask  = bert_tokenize(train_texts)
val_ids, val_mask = bert_tokenize(val_texts)
te_ids,  te_mask  = bert_tokenize(test_texts)
print(f"Done in {time.time()-t0:.1f}s  |  input_ids shape: {tr_ids.shape}")

# ── tf.data pipelines ─────────────────────────────────────────────────────────
def make_bert_ds(ids, mask, labels, shuffle=False):
    d = tf.data.Dataset.from_tensor_slices(
        ({"input_ids": ids, "attention_mask": mask}, labels)
    )
    if shuffle:
        d = d.shuffle(len(labels), seed=SEED)
    return d.batch(BERT_BATCH).prefetch(tf.data.AUTOTUNE)

bert_train_ds = make_bert_ds(tr_ids,  tr_mask,  train_labels, shuffle=True)
bert_val_ds   = make_bert_ds(val_ids, val_mask, val_labels)
bert_test_ds  = make_bert_ds(te_ids,  te_mask,  test_labels)
print("Pipelines ready.")

## 3. Model Architecture

In [ ]:
# ── Focal loss ────────────────────────────────────────────────────────────────
# Replaces cross-entropy + class_weight. Down-weights easy/confident predictions
# so training focuses on hard examples and minority classes.
# gamma=2.0 is the standard value; higher gamma = more focus on hard examples.
def focal_loss(gamma=2.0):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        probs  = tf.reduce_sum(y_pred * tf.one_hot(y_true, tf.shape(y_pred)[-1]), axis=-1)
        ce     = -tf.math.log(probs)
        weight = tf.pow(1.0 - probs, gamma)
        return tf.reduce_mean(weight * ce)
    return loss_fn

# ── RoBERTa classifier (Functional API) ──────────────────────────────────────
MODEL_NAME = "roberta-base"
print(f"Loading {MODEL_NAME}...")
roberta_base = TFRobertaModel.from_pretrained(MODEL_NAME)
roberta_base.trainable = False  # start frozen for Phase 1

inp_ids  = keras.Input(shape=(BERT_MAX_LEN,), dtype=tf.int32, name="input_ids")
inp_mask = keras.Input(shape=(BERT_MAX_LEN,), dtype=tf.int32, name="attention_mask")

# last_hidden_state: (batch, 128, 768) — per-token contextual embeddings
hidden = roberta_base(inp_ids, attention_mask=inp_mask, training=False).last_hidden_state

# CLS token (position 0) — trained by RoBERTa specifically for sequence classification
cls_out = hidden[:, 0, :]
dropped = layers.Dropout(0.3)(cls_out)
output  = layers.Dense(NUM_CLASSES, activation="softmax", name="output")(dropped)

model = keras.Model(inputs=[inp_ids, inp_mask], outputs=output, name="roberta_huffpost")
model.summary()

## 4. Training

### Phase 1 — Head only (frozen base)
Train only the CLS → Dropout → Dense(35) head at a high learning rate.
Warms up the head weights before the base is unlocked.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=focal_loss(gamma=2.0),
    metrics=["accuracy"],
)

# Monitor val_accuracy (not val_loss) — these diverge on transformer fine-tuning.
# val_loss can bottom out early while val_accuracy keeps improving.
es_p1 = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=3, restore_best_weights=True, verbose=1
)

print("── Phase 1: frozen base, head only ─────────────────────────────────────")
t0 = time.time()
hist_p1 = model.fit(
    bert_train_ds,
    validation_data=bert_val_ds,
    epochs=5,
    callbacks=[es_p1],
    verbose=1,
)
train_time_p1 = time.time() - t0
print(f"\nPhase 1 complete — {train_time_p1/60:.1f} min")

### Phase 2 — Full fine-tune (warmup + cosine decay)
Unfreeze all 125M RoBERTa parameters. Linear warmup over 1 epoch brings the LR from 0 → 2e-5,
then cosine decay reduces it smoothly to ~0 over the remaining epochs.
This prevents catastrophic forgetting while allowing full domain adaptation.

In [ ]:
roberta_base.trainable = True

# ── Warmup + cosine decay LR schedule ────────────────────────────────────────
PHASE2_EPOCHS   = 10
steps_per_epoch = len(train_labels) // BERT_BATCH
total_steps     = PHASE2_EPOCHS * steps_per_epoch
warmup_steps    = steps_per_epoch   # linear warmup over first epoch

class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, warmup_steps, total_steps):
        self.peak_lr      = float(peak_lr)
        self.warmup_steps = float(warmup_steps)
        self.total_steps  = float(total_steps)

    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.peak_lr * (step / self.warmup_steps)
        cos_decay = 0.5 * self.peak_lr * (
            1.0 + tf.cos(np.pi * (step - self.warmup_steps) /
                         (self.total_steps - self.warmup_steps))
        )
        return tf.where(step < self.warmup_steps, warmup_lr, cos_decay)

    def get_config(self):
        return {"peak_lr": self.peak_lr,
                "warmup_steps": self.warmup_steps,
                "total_steps":  self.total_steps}

lr_schedule = WarmupCosineDecay(
    peak_lr      = 2e-5,
    warmup_steps = warmup_steps,
    total_steps  = total_steps,
)

model.compile(
    optimizer=keras.optimizers.Adam(lr_schedule),
    loss=focal_loss(gamma=2.0),
    metrics=["accuracy"],
)

es_p2 = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=3, restore_best_weights=True, verbose=1
)

print("── Phase 2: full fine-tune, cosine decay LR ─────────────────────────────")
print(f"Max epochs: {PHASE2_EPOCHS}  |  Steps/epoch: {steps_per_epoch:,}  |  Warmup: {warmup_steps:,} steps")
t0 = time.time()
hist_p2 = model.fit(
    bert_train_ds,
    validation_data=bert_val_ds,
    epochs=PHASE2_EPOCHS,
    callbacks=[es_p2],
    verbose=1,
)
train_time_p2 = time.time() - t0
print(f"\nPhase 2 complete — {train_time_p2/60:.1f} min")
print(f"Total training time: {(train_time_p1 + train_time_p2)/60:.1f} min")

## 5. Evaluation

In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────────────
val_loss,  val_acc  = model.evaluate(bert_val_ds,  verbose=0)
test_loss, test_acc = model.evaluate(bert_test_ds, verbose=0)

val_preds  = np.argmax(model.predict(bert_val_ds,  verbose=0), axis=1)
test_preds = np.argmax(model.predict(bert_test_ds, verbose=0), axis=1)

val_f1  = f1_score(val_labels,  val_preds,  average="macro")
test_f1 = f1_score(test_labels, test_preds, average="macro")

print(f"Val  accuracy: {val_acc:.4f}   Val  macro-F1: {val_f1:.4f}")
print(f"Test accuracy: {test_acc:.4f}   Test macro-F1: {test_f1:.4f}")
print()
# Comparison to OPTIMAL v1: DistilBERT, GAP pooling, cross-entropy+class_weight
PREV_ACC = 0.6457
PREV_F1  = 0.5824
print("── vs. OPTIMAL v1 (DistilBERT, GAP, cross-entropy+class_weight) ─────────")
print(f"  Test accuracy  Δ: {test_acc - PREV_ACC:+.4f}  ({test_acc:.4f} vs {PREV_ACC})")
print(f"  Test macro-F1  Δ: {test_f1  - PREV_F1:+.4f}  ({test_f1:.4f} vs {PREV_F1})")

In [ ]:
# ── Training curves (Phase 1 + Phase 2 combined) ─────────────────────────────
p1_n = len(hist_p1.history["accuracy"])
p2_n = len(hist_p2.history["accuracy"])

all_acc     = hist_p1.history["accuracy"]     + hist_p2.history["accuracy"]
all_val_acc = hist_p1.history["val_accuracy"] + hist_p2.history["val_accuracy"]
all_loss    = hist_p1.history["loss"]         + hist_p2.history["loss"]
all_val_lss = hist_p1.history["val_loss"]     + hist_p2.history["val_loss"]
epochs = range(1, p1_n + p2_n + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, train_vals, val_vals, title in [
    (axes[0], all_acc,  all_val_acc, "Accuracy"),
    (axes[1], all_loss, all_val_lss, "Loss"),
]:
    ax.plot(epochs, train_vals, label="Train")
    ax.plot(epochs, val_vals,   label="Validation")
    ax.axvline(p1_n + 0.5, color="gray", linestyle="--", alpha=0.6, label="Phase 1→2")
    ax.set_title(title); ax.set_xlabel("Epoch")
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle(f"DistilBERT Fine-tuning — {NUM_CLASSES} Consolidated Classes", fontsize=13)
plt.tight_layout()
plt.savefig("optimal_curves.png", dpi=100)
plt.show()

In [ ]:
# ── Per-class F1 breakdown ────────────────────────────────────────────────────
report = classification_report(
    test_labels, test_preds,
    labels=np.arange(NUM_CLASSES),
    target_names=label_names,
    output_dict=True,
    zero_division=0,
)
class_f1   = {label_names[i]: report[label_names[i]]["f1-score"] for i in range(NUM_CLASSES)}
sorted_f1  = sorted(class_f1.items(), key=lambda x: x[1])

print(f"Per-class F1 — {NUM_CLASSES} classes (5 hardest / 5 easiest):")
for cls, score in sorted_f1[:5]:
    print(f"  HARD  {cls:<28}: {score:.3f}")
print("  ...")
for cls, score in sorted_f1[-5:]:
    print(f"  EASY  {cls:<28}: {score:.3f}")

# ── Bar chart: all classes ────────────────────────────────────────────────────
names  = [x[0] for x in sorted_f1]
scores = [x[1] for x in sorted_f1]
colors = ["#d62728" if s < 0.5 else "#2ca02c" if s > 0.75 else "#1f77b4" for s in scores]

fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(names, scores, color=colors)
ax.axvline(0.5,  color="red",   linestyle="--", alpha=0.5, label="F1 = 0.50")
ax.axvline(0.75, color="green", linestyle="--", alpha=0.5, label="F1 = 0.75")
ax.set_xlabel("F1 Score"); ax.set_title("Per-Class F1 — DistilBERT (Consolidated Labels)")
ax.legend(); plt.tight_layout()
plt.savefig("optimal_per_class_f1.png", dpi=100)
plt.show()

In [ ]:
# ── Sample misclassifications ─────────────────────────────────────────────────
wrong_idx = np.where(test_preds != test_labels)[0][:8]
print("Sample misclassifications:")
for idx in wrong_idx:
    true  = label_names[test_labels[idx]]
    pred  = label_names[test_preds[idx]]
    text  = test_texts[idx][:110]
    print(f"  True: {true:<22} → Pred: {pred}")
    print(f"  Text: {text}...")
    print()